# 6일차 예제 · 파인튜닝의 이해 / sLM 경량화와 상용 API 비교 / 성능 평가

실습 파일은 RunPod 실습 공간의 `/workspace`에 이미 올라와 있습니다. `00_시작하기.ipynb`를 먼저 실행했으면 이 노트북을 위에서부터 차례로 실행합니다. 모델(google/gemma-4-E2B-it, 약 9.5GB)은 예제 1-1을 처음 실행할 때 한 번 내려받습니다.

처음 여는 사람은 맨 아래 부록 셀을 실행하지 않습니다. 부록은 예제 1-3 (5)까지 마친 뒤 실습 공간을 다시 켰거나 커널을 다시 시작했을 때만 씁니다. 그때는 「환경 준비」 셀과 「준비 셀」(서로 다른 셀입니다)을 차례로 실행한 다음 「부록. 공간이 꺼졌을 때 복구하기」 셀을 실행합니다.

### Claude Code에 물어보는 법

셀의 코드가 궁금하면 셀을 통째로 복사해 PC의 Claude Code에 붙이고, 아래 셋 가운데 하나를 덧붙여 묻습니다. Claude Code가 알려 준 코드는 그 셀 바로 아래 새 셀에 붙여 실행합니다.

그 코드가 `base_model`, `evalset`처럼 노트북에 이미 있는 변수를 덮어썼다면, 원래 셀을 한 번 더 실행하고 다음 예제로 넘어갑니다. 받은 코드가 모델을 다시 학습하거나 GPU에 새로 올리는 코드라면 시간이 오래 걸리니, 실행하기 전에 그런 줄이 있는지 먼저 확인합니다.

1. 「이 셀에서 ○○ 줄이 하는 일을 한 문장으로 알려 줘」
2. 「○○ 값을 △△로 바꿔 실행하면 결과가 어떻게 달라지는지 알려 줘」
3. 「이 오류가 왜 났고 어떻게 고치는지 알려 줘」(오류 메시지 전문을 함께 붙입니다)

셀마다 맨 끝 주석 `# Claude Code에 물어보기:`에 그 셀에 맞는 질문을 하나씩 적어 두었습니다.

## 환경 준비

노트북을 처음 열었을 때와 커널을 다시 시작했을 때 아래 셀을 한 번 실행합니다.

필요한 라이브러리를 버전을 고정해 설치하고, 실습 파일 두 개가 있는지 확인합니다. 공간에 미리 깔린 transformers 4.57로는 오늘 모델(Gemma 4)을 올리지 못합니다. 버전을 고정해 두면 강사 화면과 같은 결과가 나옵니다.

마지막에 「있음」 두 줄이 나오면 준비가 끝난 것입니다. 「Note: you may need to restart the kernel」 안내가 나와도 커널을 다시 시작하지 않습니다.

In [ ]:
# 환경 준비: 노트북을 열면 가장 먼저 한 번 실행합니다
import os, glob, zipfile

# 결과와 상관없는 경고, 진행 막대, 로그인 안내를 숨깁니다
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['HF_HUB_VERBOSITY'] = 'error'

# 1. 실습 파일은 이미 /workspace에 있습니다. day06_data가 없을 때만 zip을 풉니다
for zip_path in sorted(glob.glob('*.zip')):
    names = zipfile.ZipFile(zip_path).namelist()
    if 'day06_data/chunks.json' in names and not os.path.isdir('day06_data'):
        zipfile.ZipFile(zip_path).extractall()
        print('풀었습니다 ·', zip_path)

# 2. 버전을 고정해 설치합니다. restart 안내가 떠도 다시 시작하지 않아도 됩니다
%pip install -q "transformers==5.18.0" "huggingface-hub<2" "peft==0.21.2"
%pip install -q "accelerate==1.15.0" "bitsandbytes==0.50.2" "anthropic==1.11.0"

# 3. 두 줄 다 「있음」이면 끝입니다. 「없음」이면 00_시작하기.ipynb를 먼저 실행합니다
for path in ['day06_data/chunks.json', 'mission_data/past_alarms.json']:
    print('있음' if os.path.exists(path) else '없음', path)
# Claude Code에 물어보기: transformers 5.18.0과 peft 0.21.2가 설치됐는지 버전을 출력하는 코드 작성해 줘

## 실습 패키지에 들어 있는 파일

| 파일 | 내용 | 사용처 |
| --- | --- | --- |
| `day06_data/day06_helpers.py` | 오늘 쓰는 함수(모델 올리기, 채점, 학습, 메모리 비우기)의 원문입니다. 함수마다 무엇을 하는지 주석이 있습니다. | 아래 준비 셀이 가져옵니다. |
| `day06_data/chunks.json` | 3~5일차 RAG 실습에 쓴 SL-FAB 사내 문서를 청크로 나눠 담은 파일입니다. 청크마다 doc_id, title, type(유형 8종), text가 들어 있습니다. | 예제 1-1이 제목 → 유형 데이터 180건(학습 144 / 평가셋 36)을 만듭니다. |
| `day06_data/b6_facts.json` | 강사가 이 노트북을 끝까지 실행해 측정한 값입니다. | 공간을 다시 켜서 앞 예제를 건너뛰었을 때만 3-1과 3-2가 이 값을 읽습니다. |
| `day06_data/api_cache6.json` | 상용 API(Claude) 답 36건을 저장한 캐시입니다. | 2-3 (2)가 API 키 없이 실행될 때만 이 파일에서 답을 꺼냅니다. |
| `lora_doctype/` (실행하면 생김) | 예제 1-3 (5)가 저장하는 LoRA 어댑터입니다. 약 10MB이고 원본 모델은 들어 있지 않습니다. | 1-4, 2-2, 2-3, 3-1이 이 폴더를 붙여 씁니다. |

## 준비 셀

오늘 쓰는 함수를 가져옵니다. 노트북을 열 때마다 한 번 실행합니다.

함수 파일 `day06_helpers`를 가져오고 난수를 고정합니다. 긴 준비 코드를 셀마다 되풀이하지 않도록 함수를 파일 하나에 모았습니다. 출력에 `장치 cuda`가 나오면 GPU를 쓸 준비가 된 것입니다.

In [ ]:
# 준비 셀: 오늘 쓰는 함수를 가져옵니다 (함수 원문은 day06_data/day06_helpers.py)
import sys, time
from collections import Counter

# 함수 파일이 있는 폴더를 먼저 찾게 하고, load_model, evaluate 등을 가져옵니다
sys.path.insert(0, 'day06_data')
from day06_helpers import *

# 난수를 42로 고정해 다시 실행해도 같은 결과가 나오게 합니다
set_seed()
print('장치', DEVICE, '· 모델', MODEL_NAME)
# Claude Code에 물어보기: day06_helpers의 함수 목록과 역할을 한 줄씩 정리해 알려 줘

## 예제 1-1. 학습 전 정확도를 측정합니다

**파인튜닝**: 이미 학습된 모델을 우리 과제의 데이터로 한 번 더 학습해 그 과제에 맞추는 일입니다. 오늘 과제는 사내 문서 제목을 보고 문서 유형을 맞히는 분류입니다.

**1. 작은 모델 내려받기**  
파라미터 51억 개짜리로, 실습 GPU 한 장에서 학습까지 할 수 있습니다.

모델은 수많은 숫자(파라미터)의 묶음입니다. GPU에 올린다는 것은 그 숫자를 GPU 메모리로 옮겨 계산할 준비를 한다는 뜻이라, 모델이 클수록 GPU 메모리를 많이 씁니다.

모델을 GPU에 올리고 파라미터 수를 집계합니다. 크기를 알아야 GPU 메모리에 들어갈지, 학습에 메모리가 얼마나 들지 추정할 수 있습니다.

「파라미터 5.1 B」가 나오면 제대로 올라간 것입니다. 처음 실행하면 모델 파일을 내려받느라 몇 분 걸리고, 두 번째부터는 금방 끝납니다.

In [ ]:
# 모델을 GPU에 올리고 파라미터 수를 집계합니다
start = time.perf_counter()
base_model = load_model()

# numel()은 텐서에 든 값의 개수를 돌려줍니다. 파라미터 텐서마다 세어 더합니다
param_count = sum(parameter.numel() for parameter in base_model.parameters())

print('파라미터', round(param_count / 1e9, 2), 'B')
print('올리는 데', round(time.perf_counter() - start, 1), '초')
# Claude Code에 물어보기: base_model.parameters()를 층 이름별로 묶어 파라미터 수 상위 5개를 집계해 줘

**2. 분류할 유형과 문서 불러오기**  
3~5일차 RAG 실습에 쓰던 그 코퍼스입니다

코퍼스는 검색 대상이 되는 사내 문서 묶음 전체이고, 청크는 긴 문서를 검색하기 좋게 잘라 둔 조각입니다. 오늘은 검색은 하지 않고 문서마다 제목과 유형(type)만 꺼내 씁니다.

청크 파일을 열어 문서마다 첫 청크 하나만 남기고, 정답이 될 유형 목록을 만듭니다. 제목을 보고 유형을 맞히는 과제라 문서마다 제목과 유형 하나씩이면 충분합니다. 문서 252건, 유형 8종이 나오면 다음 셀로 넘어갑니다.

In [ ]:
# 청크를 열고 문서마다 첫 청크 하나만 남깁니다
chunks = load_json('day06_data/chunks.json')
documents = {}
for chunk in chunks:
    documents.setdefault(chunk['doc_id'], chunk)

# 정답이 될 유형 8종
label_names = sorted({chunk['type'] for chunk in chunks})

print('문서', len(documents))
print('유형', ' / '.join(label_names))
# Claude Code에 물어보기: chunks에서 doc_id당 청크 수 분포를 집계해 출력해 줘

**3. 학습 데이터와 평가셋 분리, 프롬프트 작성**  
유형을 전부 알려 주고 한 단어로 답하라고 합니다

프롬프트는 모델에게 주는 지시와 입력을 합친 글로, `{q}` 자리에 문서 제목을 끼워 넣습니다. 평가셋은 정답을 미리 알고 있는 문제 묶음으로, 학습 전과 후에 같은 문제로 채점해 비교하는 데 씁니다.

제목과 유형으로 데이터를 만들어 학습 데이터 144건과 평가셋 36건으로 나누고, 프롬프트를 정해 둡니다. 평가셋은 채점에만 쓰고 학습에는 넣지 않아야, 학습한 뒤에도 처음 보는 문제로 점수를 측정할 수 있습니다.

출력된 프롬프트의 `{q}` 자리에 평가셋 첫 제목이 들어가 있으면 됩니다.

In [ ]:
# 제목(q) → 유형(a) 데이터를 만들어 학습 데이터와 평가셋으로 나눕니다
all_rows, train_rows, evalset = make_title_rows(documents, label_names)

prompt_template = ('다음 사내 문서 제목이 어느 유형인지 한 낱말로만 답하라.\n'
                   '유형: ' + ' / '.join(label_names) + '\n\n제목: {q}\n유형:')

# evaluate와 train_lora가 이 프롬프트를 씁니다
set_task(prompt_template, label_names)
print(prompt_template.format(q=evalset[0]['q']))
# Claude Code에 물어보기: evalset 36건에 label_names 8종이 모두 들어 있는지 확인해 줘

**4. 평가셋 36문항 채점**  
맞은 개수와 형식을 지킨 개수를 따로 집계합니다

정확도는 맞힌 문항의 비율로, 36문항 가운데 29문항을 맞히면 29 ÷ 36 = 80.6%입니다.

학습 전 모델로 평가셋을 채점해 정확도와 형식 준수율을 출력합니다. 이 점수가 기준선이고, 학습한 뒤 같은 평가셋으로 다시 측정해 비교합니다.

In [ ]:
# evaluate()는 문항마다 답을 생성해 첫 단어로 채점합니다
before_result = evaluate(base_model, evalset)

print('정확도', before_result['accuracy'], '%')
print('형식 준수', before_result['format_rate'], '%')
# Claude Code에 물어보기: evaluate()가 답의 첫 단어로 채점하는 코드를 day06_helpers에서 찾아 알려 줘

**출력 읽는 법**: 정확도는 36문항 중 답의 첫 단어가 정답 유형과 같은 비율, 형식 준수율은 첫 단어가 유형 8종 가운데 하나인 비율입니다. 형식은 지켰는데 틀린 답이 많으면 모델이 유형을 구분하는 기준이 우리 회사 기준과 다른 것이고, 학습 데이터로 그 기준을 맞출 수 있습니다.

**5. 모델의 답 확인**  
모델의 답과 정답을 나란히 놓습니다

틀린 문항만 골라 「모델의 답 ← 정답 · 제목 앞부분」으로 출력합니다. 점수만으로는 모델이 무엇을 헷갈리는지 모릅니다. 틀린 답을 읽어 보고 학습으로 개선할 만한 문제인지 판단합니다.

In [ ]:
# 모델의 답 ← 정답, 제목 앞부분
for answer, row in zip(before_result['answers'], evalset):
    if answer != row['a']:
        print(repr(answer), '←', row['a'], '·', row['q'][:16])
# Claude Code에 물어보기: before_result 오답에서 「답 → 정답」 쌍별 빈도를 집계해 줘

**여기까지 확인**
- 모델을 올렸습니다: 파라미터 5.1B
- 두 지표를 확인했습니다: 정확도와 형식 준수율
- 틀린 답을 열어 봤다: 어느 유형과 헷갈리는지

## 예제 1-2. 학습 데이터를 만듭니다

**1. 코퍼스에서 학습 데이터 추출**  
문서 제목이 질문이고 유형이 정답입니다

예제 1-1 (3)에서 만든 데이터의 건수와 앞 세 건을 「유형 ← 제목」으로 출력합니다. 학습 데이터가 어떻게 생겼는지 보면 모델이 무엇을 배우는지 알 수 있습니다. 문서 252건 가운데 같은 제목은 한 건만 남겨 「총 180 건」이 됩니다.

In [ ]:
# 예제 1-1 (3)에서 만든 행 가운데 앞 세 건을 출력합니다
print('총', len(all_rows), '건')

for row in all_rows[:3]:
    print(' ', row['a'], '←', row['q'][:22])
# Claude Code에 물어보기: make_title_rows가 252건을 180건으로 줄이는 중복 제거 기준을 알려 줘

**2. 학습 데이터와 평가셋 분리**  
학습에 쓴 제목은 평가셋에 넣지 않습니다

예제 1-1 (3)에서 나눈 학습 데이터와 평가셋의 건수를 출력합니다. 학습 144, 평가셋 36이 나옵니다. 평가셋이 36문항이면 한 문항이 약 2.8%p입니다.

In [ ]:
print('학습', len(train_rows))
print('평가셋', len(evalset))
# Claude Code에 물어보기: train_rows와 evalset에 같은 제목이 겹치지 않는지 검사하는 코드 작성해 줘

**3. 유형별 분포 확인**  
여덟 유형이 골고루 있는지 확인합니다

유형별 건수를 집계해 많은 순서로 출력합니다. 어느 유형이 너무 적으면 모델이 그 유형을 잘 배우지 못합니다. 가장 많은 품질이슈(50건)와 가장 적은 안전규정(5건)이 열 배 차이 납니다.

In [ ]:
# 유형마다 몇 건인지 많은 순서로 집계합니다
label_counts = Counter(row['a'] for row in all_rows)

for label, count in label_counts.most_common():
    print(f'{label:8s}', count)
# Claude Code에 물어보기: label_counts가 열 배 차이일 때 쓸 오버샘플링 코드 예를 알려 줘

**4. 학습 문장 만들기**  
채팅 형식으로 감싼 프롬프트 뒤에 정답을 붙인 것이 학습 데이터입니다

토큰은 모델이 글을 읽고 쓰는 단위 조각이며, 단어 하나가 토큰 여러 개로 나뉘기도 합니다. 모델이 한 번에 읽는 길이와 API 요금이 모두 토큰 수로 정해집니다.

학습 데이터 한 건을 채팅 형식 문장으로 만들어 끝부분을 출력하고, 토크나이저(글을 토큰으로 자르는 도구)로 자른 토큰 수를 계산합니다. 채팅 템플릿은 누가 말하는 차례인지 표시해 둔 형식이고, 입력을 모델이 학습할 때 본 이 형식에 맞춰 넣어야 제대로 답합니다. `<|turn>model` 다음 줄에 정답 유형과 EOS 토큰 `<eos>`가 붙어 있으면 됩니다.

In [ ]:
# build_text()는 채팅 형식 프롬프트 뒤에 정답과 EOS 토큰(<eos>)을 붙입니다
train_text = build_text(train_rows[0])
print(train_text[-70:])

# 토크나이저로 자른 토큰 수를 계산합니다
token_ids = tokenizer(train_text, add_special_tokens=False)['input_ids']
print('토큰', len(token_ids))
# Claude Code에 물어보기: train_rows 전체의 토큰 수를 집계해 최대값이 128을 넘는지 확인해 줘

**여기까지 확인**
- 180건을 나눴습니다: 144와 36
- 유형 분포를 확인했습니다: 50건과 5건, 열 배 차이
- 학습 문장을 확인했습니다: 채팅 형식, 정답, EOS 토큰

## 예제 1-3. LoRA로 학습합니다

- **LoRA**(Low-Rank Adaptation): 원본 가중치는 고정하고 작은 행렬 두 개를 덧붙여 그것만 학습하는 방식입니다. 학습할 값이 적어 GPU 한 장으로도 충분하고, 결과는 작은 어댑터 파일로 따로 저장합니다.
- **에포크**: 학습 데이터 전체를 처음부터 끝까지 한 번 학습하는 단위입니다. **배치**: 한 번에 묶어 학습하는 건수입니다.
- **학습률**: 한 번에 가중치를 얼마나 크게 업데이트할지 정하는 값입니다. **r**: 덧붙이는 행렬의 rank이고, 클수록 학습할 값이 늘어납니다.

**1. 학습 실행**  
r 8, 3에포크, 배치 4, 학습률 2e-4

1-1에서 올린 모델을 내리고 `train_lora()`로 학습합니다. 원본 가중치는 고정하고 LoRA 행렬만 업데이트하므로 GPU 한 장으로 학습할 수 있습니다. 에포크마다 손실이 한 줄씩 나오고, 마지막에 학습 시간이 나옵니다.

In [ ]:
# train_lora()가 모델을 새로 올려 LoRA를 붙이고 학습합니다
# 1-1에서 올린 모델은 먼저 내립니다
free_memory('base_model')

model, history = train_lora(train_rows, lora_rank=8, epochs=3)
print('학습 시간', history['seconds'], '초')
# Claude Code에 물어보기: history의 에포크별 손실을 텍스트 그래프로 표시해 줘

**출력 읽는 법**: 에포크마다 손실이 한 줄씩 나옵니다. 손실은 모델의 답이 정답과 얼마나 다른지를 숫자 하나로 나타낸 값이라, 에포크가 지날수록 내려가면 학습이 되고 있다는 뜻입니다. 다만 손실은 학습 데이터 기준이므로 좋아졌는지는 평가셋 정확도로 확인합니다.

**2. 학습한 파라미터 비율 계산**  
원본 가중치는 고정하고 LoRA 행렬만 학습했습니다

LoRA로 학습한 값이 전체에서 차지하는 비율을 계산합니다. 2.68M, 전체의 0.052%가 나옵니다. 이만큼만 업데이트하므로 메모리와 시간이 적게 듭니다.

In [ ]:
# 학습한 파라미터와 전체 파라미터를 비교합니다
trained = history['trainable_params']
total = sum(parameter.numel() for parameter in model.parameters())

print('학습한 파라미터', round(trained / 1e6, 2), 'M')
print('전체 대비 비율', round(trained / total * 100, 3), '%')
# Claude Code에 물어보기: trained 2.68M이 r=8과 q, k, v, o 네 곳에서 어떻게 계산되는지 알려 줘

**3. 같은 평가셋으로 재측정**  
예제 1-1과 같은 평가셋 36문항입니다

학습한 모델로 예제 1-1과 같은 평가셋을 같은 방법으로 채점합니다. 그래야 점수 차이를 학습 효과로 볼 수 있습니다. 강사 실행에서는 정확도가 80.6%에서 94.4%로 오르고 형식 준수율이 100%였습니다. GPU에 따라 한두 문항(한 문항이 약 2.8%p) 다를 수 있고, 한 건당 시간도 공간마다 조금 다릅니다.

In [ ]:
# 예제 1-1과 같은 evaluate()와 같은 평가셋으로 측정합니다
after_result = evaluate(model, evalset)

print('정확도', after_result['accuracy'], '%')
print('형식 준수', after_result['format_rate'], '%')
print('한 건당', after_result['seconds_per_question'], '초')
# Claude Code에 물어보기: evaluate() 결과를 evalset 유형별 정확도로 집계하는 방법 알려 줘

**4. 남은 오답 확인**  
학습 뒤에 남은 오답입니다

학습 뒤에도 틀린 문항만 같은 형식으로 출력합니다. 점수가 올랐어도 남은 오답을 읽어 보면 어떤 유형끼리 헷갈리는지 보이고, 다음 개선 방향도 거기서 나옵니다. 학습 전보다 오답 수가 줄었으면 됩니다.

In [ ]:
for answer, row in zip(after_result['answers'], evalset):
    if answer != row['a']:
        print(repr(answer), '←', row['a'], '·', row['q'][:16])
# Claude Code에 물어보기: after_result 오답 문항의 제목을 잘리지 않게 전체 출력해 줘

**5. 어댑터 저장**  
어댑터는 LoRA 행렬만 담은 파일입니다. 원본 모델은 그대로 둡니다

어댑터를 `lora_doctype` 폴더에 저장하고, 폴더 크기를 원본 모델 크기와 함께 MB로 출력합니다. 원본 모델은 어디서나 같으므로 작은 어댑터 파일만 주고받으면 학습 결과를 다른 곳에서 다시 붙여 쓸 수 있습니다. 어댑터는 10.3MB로 원본의 약 천 분의 일입니다.

In [ ]:
# save_pretrained()는 LoRA 행렬만 폴더에 씁니다
model.save_pretrained('lora_doctype')

# 폴더 안 파일 크기를 더합니다
adapter_bytes = 0
for name in os.listdir('lora_doctype'):
    adapter_bytes += os.path.getsize(os.path.join('lora_doctype', name))

print('어댑터 크기', round(adapter_bytes / 1024**2, 1), 'MB')
print('원본 모델  ', round(param_count * 2 / 1024**2), 'MB (bf16 · 값 하나 2바이트)')
# Claude Code에 물어보기: adapter_config.json의 r과 target_modules를 출력해 줘

**여기까지 확인**
- 에포크마다 손실이 내려갔습니다.
- 학습한 파라미터는 전체의 0.05%입니다.
- 같은 평가셋 36문항에서 정확도가 올랐습니다.

## 예제 1-4. 학습 뒤에 망각과 환각을 점검합니다

- **망각**: 새 과제를 학습하면서 원래 잘 답하던 질문에 제대로 답하지 못하게 되는 현상입니다. 크게 잃으면 파국적 망각(catastrophic forgetting)이라고 합니다. 학습한 모델을 쓰기 전에 원래 능력이 남아 있는지 점검합니다.
- **환각** 다시 보기: 근거가 없는 내용을 사실처럼 지어낸 답입니다.

**1. 학습 전후 모델 준비**  
저장한 어댑터를 붙이면 학습 후, 떼면 학습 전 모델입니다

1-3의 모델을 내리고, 원본 모델에 저장해 둔 어댑터를 붙여 다시 올립니다. 어댑터를 붙였다 떼는 것만으로 모델 하나에서 학습 전후를 번갈아 쓸 수 있습니다. 「어댑터를 붙인 모델 준비」 한 줄이 나오면 됩니다.

In [ ]:
from peft import PeftModel

# 1-3의 모델을 내립니다
free_memory('model')

# 원본 모델에 저장한 어댑터를 붙입니다
tuned_model = PeftModel.from_pretrained(load_model(), 'lora_doctype')
print('어댑터를 붙인 모델 준비 · 떼면 학습 전 모델입니다')
# Claude Code에 물어보기: tuned_model의 어댑터를 merge_and_unload()로 합치는 방법 알려 줘

**2. 학습과 무관한 질문 입력**  
분류와 상관없는 질문 세 개를 정해 두고 하나를 먼저 입력합니다

질문 세 개 가운데 세 번째 질문을 먼저 넣어 답의 앞부분을 출력합니다.

학습한 과제만 잘하게 되고 원래 하던 답을 잃었다면 망각입니다. 이것을 살피려고 분류와 상관없는 질문을 따로 준비합니다.

In [ ]:
# generate_answer()는 프롬프트 틀 없이 질문을 그대로 넣어 답을 받습니다
questions = ['대한민국의 수도는 어디인가요?',
             '물이 끓는 온도는 몇 도인가요?',
             '1부터 10까지 더하면 얼마인가요?']

print(generate_answer(tuned_model, questions[2])[:44])
# Claude Code에 물어보기: generate_answer()의 최대 생성 토큰과 디코딩 설정을 helpers에서 찾아 알려 줘

**3. 세 질문의 전후 답 비교**  
`disable_adapter()` 안에서는 어댑터를 뗀 학습 전 모델이 답합니다

질문마다 학습 전 답과 학습 후 답을 「전」, 「후」 짝으로 출력합니다. 같은 질문의 답을 나란히 놓으면 무엇을 잃었는지 알아채기 쉽습니다.

In [ ]:
for question in questions:
    # 이 안에서는 어댑터를 뗀 학습 전 모델이 답합니다
    with tuned_model.disable_adapter():
        print('전', generate_answer(tuned_model, question)[:30])
    print('후', generate_answer(tuned_model, question)[:30])
# Claude Code에 물어보기: questions에 문서 제목 하나를 더해 전후 답을 비교하는 코드 작성해 줘

**4. 핵심 답 유지 여부 집계**  
답 문장은 매번 조금씩 달라서 글자 그대로 비교하지 않습니다

질문마다 꼭 들어가야 할 답(서울, 100, 55)을 정해 두고, 학습 전후의 답에 그 말이 들어 있는 질문 수를 집계합니다. 학습 후 수가 학습 전보다 작으면 망각입니다.

In [ ]:
# 질문마다 핵심 답이 답에 들어 있는지 학습 전후로 집계합니다
key_answers = ['서울', '100', '55']
kept_before, kept_after = count_kept_answers(tuned_model, questions, key_answers)

print('핵심 답을 담은 질문 · 전', kept_before, '/', len(questions))
print('                    후', kept_after, '/', len(questions))
# Claude Code에 물어보기: count_kept_answers()가 key_answers 포함 여부를 판정하는 방식을 알려 줘

**출력 읽는 법**: 「전」은 어댑터를 뗀 학습 전 모델, 「후」는 붙인 모델입니다. 「후」가 「전」보다 작으면 학습하면서 원래 하던 답을 잃은 것, 즉 망각입니다.

**5. 없는 문서 질문으로 환각과 답의 변동 확인**  
코퍼스에 없는 문서 번호를 묻고, 같은 질문을 샘플링으로 세 번 생성합니다

코퍼스에 없는 문서 번호를 묻고, 시드를 바꿔 가며 샘플링으로 답을 세 번 생성합니다. 오늘 파인튜닝은 제목을 보고 유형을 고르는 것만 가르쳤고 문서 내용은 넣지 않았으니, 모델은 이 문서를 모릅니다. 그래도 그럴듯한 답을 지어낼 수 있습니다.

샘플링은 가장 그럴듯한 단어만 고르지 않고 확률에 따라 골라 생성하는 방식이라 같은 질문에도 답이 달라질 수 있습니다. 「코퍼스에 없음」 다음에 답 세 줄이 나오고, 샘플링이라 답이 서로 다를 수 있습니다.

In [ ]:
# SOP-ETCH-099는 코퍼스에 없는 문서 번호입니다
question = 'SL-FAB 의 SOP-ETCH-099 문서는 무엇을 다루나요? 한 문장으로 답하세요.'
in_corpus = any('SOP-ETCH-099' in chunk['text'] for chunk in chunks)
print('코퍼스에 있음' if in_corpus else '코퍼스에 없음')

# 같은 질문을 샘플링으로 세 번 생성합니다. 시드를 0, 1, 2로 정해 두어 다시 실행해도 같은 세 답이 나옵니다
for attempt in range(3):
    torch.manual_seed(attempt)
    print(attempt + 1, generate_answer(tuned_model, question, sample=True)[:46])
# Claude Code에 물어보기: sample=True일 때 temperature 값에 따라 답이 어떻게 달라지는지 알려 줘

**여기까지 확인**
- 전후 모델이 둘 다 있다: 어댑터를 붙였다 뗐다
- 질문 셋을 전후로 집계했다: 핵심 답 수
- 없는 문서를 물어도 답이 나옵니다: 샘플링이라 세 답이 서로 다를 수 있습니다

## 예제 2-1. 모델이 쓰는 GPU 메모리를 계산합니다

- **정밀도**: 가중치 값 하나를 몇 바이트로 담는지입니다. fp32는 4바이트, bf16은 2바이트, int8은 1바이트, int4는 0.5바이트입니다.
- **양자화**: 정밀도를 낮춰 모델을 더 적은 메모리에 올리는 일입니다. 메모리를 아끼는 대신 정확도가 떨어질 수 있습니다.

앞 예제의 모델을 모두 내리고, 지금 GPU에 올라간 크기를 출력합니다. 다른 모델이 남아 있으면 측정값이 부풀려지므로 먼저 비웁니다. 「지금 GPU 에 올라간 것 0.0 GB」가 나오면 됩니다.

In [ ]:
# 앞 예제의 모델을 내리고 빈 GPU에서 측정을 시작합니다
free_memory('tuned_model', 'model', 'base_model')
print('지금 GPU 에 올라간 것', round(gpu_memory_gb(), 1), 'GB')
# Claude Code에 물어보기: free_memory()가 gc와 torch.cuda.empty_cache를 어떻게 쓰는지 알려 줘

**1. 파라미터 수로 메모리 계산**  
모델을 올리지 않고 구성 파일(config)만 읽어 파라미터 수를 집계합니다

구성 파일로 값이 비어 있는 모델 모양만 만들어 파라미터 수를 집계하고, 정밀도 네 가지의 크기를 GB로 출력합니다. 크기는 파라미터 수 × 값 하나의 바이트 수라서 모델을 올리지 않고도 계산됩니다. fp32, bf16, int8, int4로 내려가며 값 하나의 바이트 수가 반이 될 때마다 크기도 반이 됩니다.

In [ ]:
from transformers import AutoConfig, AutoModelForCausalLM

model_config = AutoConfig.from_pretrained(MODEL_NAME)

# 값 없이 모양만 만듭니다. 메모리를 쓰지 않습니다
with torch.device('meta'):
    empty_model = AutoModelForCausalLM.from_config(model_config)
param_count = sum(parameter.numel() for parameter in empty_model.parameters())
print('파라미터', round(param_count / 1e9, 2), 'B')

# 정밀도마다 값 하나의 바이트 수를 곱합니다
precision_bytes = [('fp32', 4), ('bf16', 2), ('int8', 1), ('int4', 0.5)]
for precision, bytes_per_value in precision_bytes:
    print(precision, round(param_count * bytes_per_value / 1024**3, 1), 'GB')
# Claude Code에 물어보기: torch.device('meta')로 만든 empty_model이 메모리를 안 쓰는 이유 알려 줘

**2. GPU에 올려 실측**  
계산한 값과 맞는지 GPU가 잡은 메모리로 확인합니다

모델을 bf16으로 올려 계산값과 실측값을 나란히 출력한 뒤 내립니다. 계산이 실측과 맞으면 다른 모델의 크기도 계산으로 믿고 쓸 수 있습니다. 둘 다 9.5GB 근처로 나오면 됩니다.

In [ ]:
# bf16으로 올린 뒤 GPU에 잡힌 크기를 읽습니다
measured_model = load_model()

print('계산', round(param_count * 2 / 1024**3, 1), 'GB')
print('실측', round(gpu_memory_gb(), 1), 'GB')

free_memory('measured_model')
# Claude Code에 물어보기: 계산 9.5GB와 gpu_memory_gb() 실측이 다를 때 생길 수 있는 원인 알려 줘

**3. GPU 한도 확인**  
현업에서 쓰는 크기로 같은 계산을 합니다

GPU 메모리의 80%를 한도로 두고, Gemma 4 네 크기마다 bf16, int8, int4 순서로 계산해 한도 안에 처음 들어가는 정밀도를 출력합니다. 답을 생성하는 동안에도 앞 토큰의 계산 결과(KV 캐시)를 담을 메모리가 필요해서 20%를 남깁니다. 같은 계산을 함수로 묶은 것이 helpers의 `fit_precision()`이고, 실습 미션 2의 1단계도 같은 계산입니다.

In [ ]:
# 한도 = GPU 메모리의 80% (20%는 KV 캐시와 실행 여유분)
gpu_gb = gpu_total_gb()
limit_gb = gpu_gb * 0.8
print('GPU', round(gpu_gb, 1), 'GB · 한도', round(limit_gb, 1), 'GB')

# 후보 모델(파라미터 수와 이름)과 정밀도(값 하나의 바이트 수)
candidates = [(5.1, 'E2B 5.1B'), (8.0, 'E4B 8.0B'), (12.0, '12B'), (31.3, '31B')]
precisions = [('bf16', 2), ('int8', 1), ('int4', 0.5)]

# 모델마다 bf16 → int8 → int4 순서로 한도 안에 처음 들어가는 정밀도를 찾습니다
for params_billion, name in candidates:
    first_fit = '없음'
    for precision, bytes_per_value in precisions:
        size_gb = params_billion * 1e9 * bytes_per_value / 1024**3
        if size_gb <= limit_gb:
            first_fit = precision
            break
    print(name, '처음 들어가는 정밀도 ·', first_fit)
# Claude Code에 물어보기: candidates에 27B를 추가하고 한도 판정을 표로 출력해 줘

**4. 학습할 때 더 드는 메모리 계산**  
추론과 학습에 드는 메모리가 다릅니다

bf16 가중치와, 전체 파인튜닝 때 더 붙는 기울기와 옵티마이저 메모리를 GB로 출력합니다. 기울기는 가중치마다 업데이트할 방향과 크기이고, 옵티마이저 상태는 AdamW가 값마다 두는 보조 값 두 개입니다. 가중치 9.5GB에 기울기 9.5GB와 옵티마이저 상태 38.0GB가 더 붙어, 24GB GPU로는 전체 파인튜닝을 할 수 없습니다.

In [ ]:
# 전체 파인튜닝: 기울기는 가중치와 같은 2바이트, AdamW는 값마다 fp32 두 개(8바이트)
weights_gb = param_count * 2 / 1024**3
print('가중치(bf16)            ', round(weights_gb, 1), 'GB')
print('전체 파인튜닝이면 + 기울기 ', round(weights_gb, 1), 'GB')
print('                 + 옵티마이저 ', round(param_count * 8 / 1024**3, 1), 'GB')
print('LoRA 는 학습하는 0.05% 에만 기울기와 옵티마이저 상태가 붙습니다')
# Claude Code에 물어보기: fp32 전체 파인튜닝이면 가중치, 기울기, 옵티마이저 합계를 계산해 줘

**5. QLoRA 메모리 계산**  
원본을 4비트로 고정해 올리고 LoRA만 학습하면 가중치가 차지하는 양이 4분의 1로 줄어듭니다

전체 파인튜닝, LoRA, QLoRA의 가중치와 학습용 메모리를 더해 한 줄씩 출력합니다. QLoRA는 원본 모델을 4비트로 양자화해 올린 채 LoRA만 학습하는 방법이라, 같은 GPU로 더 큰 모델을 학습할 수 있습니다. 실제로 4비트로 학습해 보는 셀은 노트북 끝 부록에 있습니다.

합계는 전체 파인튜닝 57.0GB, LoRA 9.5GB, QLoRA 2.4GB입니다. 실제로 4비트로 올려 보면 임베딩 표처럼 4비트로 줄지 않는 부분이 있어 이보다 큽니다(예제 2-2의 int4 6.4GB). 배치와 문장 길이에 비례하는 활성화값(activation)은 이 합계에서 빠져 있습니다. 강의안 141장이 같은 계산입니다.

In [ ]:
# 세 학습 방법의 메모리 추정: 가중치 + 학습용으로 더 붙는 양 (활성화값은 뺍니다)
# 전체 파인튜닝은 가중치마다 기울기 2 + 옵티마이저 8바이트
# LoRA와 QLoRA는 LoRA 행렬의 값마다 16바이트
#   = 값 4(PEFT가 fp32로 둠) + 기울기 4 + 옵티마이저 8
lora_values = 2_678_784   # 예제 1-3 (2)에서 집계한 LoRA 행렬의 값(r 8, 어텐션 네 곳)
plans = [('전체 파인튜닝', param_count * 2, param_count * 10),
         ('LoRA',        param_count * 2, lora_values * 16),
         ('QLoRA',       param_count * 0.5, lora_values * 16)]
for method, weight_bytes, extra_bytes in plans:
    weight_gb = weight_bytes / 1024**3
    extra_gb = extra_bytes / 1024**3
    total_gb = (weight_bytes + extra_bytes) / 1024**3
    print(f'{method:6s} 가중치 {weight_gb:5.1f} GB + 학습용 {extra_gb:5.2f} GB',
          f'= {total_gb:5.1f} GB')
# Claude Code에 물어보기: plans에 활성화값(배치 4, 128 토큰) 추정치를 추가해 계산해 줘

**여기까지 확인**
- 정밀도 4종의 크기를 계산했습니다: fp32, bf16, int8, int4
- 계산과 실측이 가깝다: bf16
- 한도 판단이 나왔다: 어느 모델이 어디까지 줄여야 올라가는지

## 예제 2-2. 정밀도를 변경해 측정합니다

**1. bf16 기준 측정**  
예제 1-3에서 저장한 어댑터를 붙여 올립니다

어댑터를 붙인 모델을 bf16으로 올려 메모리, 건당 시간, 정확도, 형식 준수율을 한 줄로 출력합니다. 다음 셀에서 int8, int4로 측정한 값과 비교할 기준입니다.

In [ ]:
# measure_precision()은 정밀도 하나로 올려 메모리, 정확도, 건당 시간을 측정하고 내립니다
result = measure_precision('bf16', 'lora_doctype', evalset)

# 정밀도 이름 → 측정 결과
precision_results = {'bf16': result}

print(f"bf16 {result['memory_gb']} GB",
      f"· 건당 {result['seconds_per_question']}초",
      f"· 정확도 {result['accuracy']} · 형식 {result['format_rate']}")
# Claude Code에 물어보기: measure_precision()이 memory_gb를 어느 시점에 측정하는지 알려 줘

**2. int8, int4 측정**  
정밀도만 변경하고 나머지는 같습니다. 양자화는 bitsandbytes 라이브러리로 하며 CUDA GPU에서 실행됩니다

int8과 int4로 같은 측정을 해 한 줄씩 출력합니다. 어디까지 줄일지는 메모리를 줄인 만큼 정확도를 얼마나 잃는지 보고 정합니다. int4에서 정확도가 크게 떨어집니다.

In [ ]:
for precision in ['int8', 'int4']:
    result = measure_precision(precision, 'lora_doctype', evalset)
    precision_results[precision] = result

    print(f"{precision} {result['memory_gb']} GB",
          f"· 건당 {result['seconds_per_question']}초",
          f"· 정확도 {result['accuracy']} · 형식 {result['format_rate']}")
# Claude Code에 물어보기: int4에서 정확도가 크게 떨어지는 이유를 bitsandbytes 기준으로 알려 줘

**3. 정밀도별 속도 비교**  
정밀도마다 건당 시간을 나란히 봅니다

정밀도마다 메모리와 건당 시간을 나란히 출력합니다. 정밀도를 낮추면 빨라지는지는 장비와 라이브러리에 따라 달라서 측정해 봐야 압니다. 강사 실행(RTX 4090)에서는 메모리는 줄었지만 int8이 가장 느렸습니다. 내 표에서 순서가 달라도 잘못 실행한 것은 아닙니다.

In [ ]:
for precision, result in precision_results.items():
    print(f"{precision:5s} 메모리 {result['memory_gb']:5.1f} GB",
          f"· 건당 {result['seconds_per_question']}초")

print('메모리는 줄고, 시간은 정밀도마다 다릅니다')
# Claude Code에 물어보기: precision_results를 bf16 대비 메모리 절감률(%)로 계산해 표시해 줘

**4. 정확도 기준으로 정밀도 선택**  
정확도 기준을 먼저 정하고 그 기준으로 판단합니다

정확도 기준을 90%로 정하고 정밀도마다 통과인지 미달인지 출력합니다. 기준을 넘는 정밀도 가운데 가장 작은 것을 고르면 됩니다. bf16과 int8은 통과, int4는 미달로 나옵니다.

In [ ]:
# 이 과제에서 지키기로 한 정확도 기준: 측정 전에 정합니다
accuracy_threshold = 90.0

for precision, result in precision_results.items():
    verdict = '통과' if result['accuracy'] >= accuracy_threshold else '미달'
    print(f"{precision:5s} 정확도 {result['accuracy']:5.1f}",
          f"· 기준 {accuracy_threshold} · {verdict}")

print('기준을 지키는 가장 작은 정밀도를 고릅니다')
# Claude Code에 물어보기: accuracy_threshold를 85.0으로 변경하면 판정이 어떻게 바뀌는지 출력해 줘

**여기까지 확인**
- 정밀도 세 가지의 메모리를 측정했습니다: bf16, int8, int4
- 정확도를 비교했다: 학습한 모델로
- 기준으로 판단했다: 정확도 기준 90%

## 예제 2-3. 파인튜닝한 sLM과 상용 API를 비교합니다

**sLM**(small Language Model): 파라미터가 수십억 개 안팎인 작은 언어 모델입니다. GPU 한 장에 올려 회사 안에서 직접 실행하므로 데이터를 밖으로 보내지 않아도 됩니다.

**1. 로딩 시간과 응답 시간 측정**  
올리는 시간과 답하는 시간을 따로 측정합니다

어댑터를 붙인 모델을 올리는 시간과 한 문항에 답하는 시간을 따로 측정하고, 요청마다 모델을 올린다면 건당 몇 초가 걸리는지 출력합니다. 요청이 올 때마다 새로 올리면 답하는 시간보다 올리는 시간이 훨씬 깁니다. 그래서 모델은 한 번 올려 두고 계속 씁니다.

In [ ]:
from peft import PeftModel

# 원본 + 어댑터를 올리는 시간을 측정합니다
start = time.perf_counter()
tuned_model = PeftModel.from_pretrained(load_model(), 'lora_doctype')
load_seconds = time.perf_counter() - start

# 한 문항만 답해 봅니다
one_question = evaluate(tuned_model, evalset[:1])
print('올리는 데  ', round(load_seconds, 1), '초')
print('한 건 답하기', one_question['seconds_per_question'], '초')

# 요청마다 올리고 답하면 걸리는 시간
per_request = load_seconds + one_question['seconds_per_question']
print('요청마다 올리면 건당', round(per_request, 1), '초 · 올려 두고 씁니다')
# Claude Code에 물어보기: load_seconds가 긴 원인과 줄이는 방법을 알려 줘

**2. 상용 API로 같은 평가셋 측정**  
Claude API로 36문항을 채점합니다

API 호출은 우리 코드가 인터넷으로 회사 밖 모델인 Claude에 질문을 보내고 답을 받는 일입니다. 호출하려면 개인 키가 필요하고, 요금은 보낸 입력 토큰과 받은 출력 토큰 수로 매겨집니다.

평가셋 36문항을 Claude API에 보내 답을 받고, 호출 방식과 정확도, 건당 시간을 출력합니다. 파인튜닝한 작은 모델(sLM)이 큰 상용 API에 비해 어느 수준인지 비교합니다.

**키를 넣으려면**: API 키는 2일차에 안내한 개인 키를 씁니다. 전달 경로는 별도로 공지하며, 1인 한도는 300달러이고 교육이 끝나면 회수합니다.

1. 아래 셀을 실행하면 셀 바로 아래에 키를 입력하는 칸이 나타납니다. 입력한 키는 화면과 노트북 파일에 남지 않습니다.
2. 키가 없으면 입력 칸을 비워 두고 Enter 키만 눌러도 이 예제를 끝까지 진행합니다. 이때는 `day06_data/api_cache6.json`에 저장해 둔 강사 실측 답으로 같은 계산을 합니다.

In [ ]:
# ClaudeCaller는 키를 입력받고, 키가 없으면 저장해 둔 답을 꺼냅니다
claude = ClaudeCaller('day06_data/api_cache6.json', model_name='claude-opus-5-5')
facts = load_json('day06_data/b6_facts.json')

# 36문항을 보내고 답의 첫 단어만 남깁니다(끝의 마침표는 뗍니다)
start = time.perf_counter()
claude_answers = []
for row in evalset:
    answer_text = claude.ask(prompt_template.format(q=row['q']))
    claude_answers.append(first_word(answer_text).strip('.'))

# 키가 없으면 강사 실측 건당 시간을 씁니다
if claude.live:
    api_seconds = (time.perf_counter() - start) / len(evalset)
else:
    api_seconds = facts['api']['per_q']

answer_pairs = zip(claude_answers, evalset)
api_correct = sum(answer == row['a'] for answer, row in answer_pairs)
api_accuracy = round(api_correct / len(evalset) * 100, 1)

print('호출 방식', '실제 호출' if claude.live else '저장한 답(키 없음)')
print('상용 API 정확도', api_accuracy, '%')
print('상용 API 건당  ', round(api_seconds, 2), '초')
# Claude Code에 물어보기: ClaudeCaller가 api_cache6.json에서 답을 꺼내는 조건을 알려 줘

**3. 파인튜닝한 sLM 측정**  
문항 수를 맞춰야 비교할 수 있습니다

파인튜닝한 sLM으로 같은 36문항을 채점해 정확도와 건당 시간을 출력합니다.

In [ ]:
slm_result = evaluate(tuned_model, evalset)

print('sLM 정확도', slm_result['accuracy'], '%')
print('sLM 건당  ', slm_result['seconds_per_question'], '초')
# Claude Code에 물어보기: slm_result와 claude_answers가 둘 다 틀린 문항만 출력해 줘

**4. 정확도와 응답 시간 비교**  
정확도와 시간을 같이 봅니다

Claude API와 Gemma E2B의 정확도와 건당 시간을 두 줄 표로 출력하고, sLM이 몇 배 빠른지 계산합니다. 어느 쪽을 쓸지는 정확도 하나만으로 정하지 않고 속도와 함께 따집니다.

In [ ]:
print('            정확도  건당시간')
print(f"Claude API {api_accuracy:6.1f} % {api_seconds:6.2f}초")
print(f"Gemma E2B  {slm_result['accuracy']:6.1f} %",
      f"{slm_result['seconds_per_question']:6.2f}초")

speedup = api_seconds / slm_result['seconds_per_question']
print('빠르기', round(speedup, 1), '배')
# Claude Code에 물어보기: 정확도, 건당시간 표에 1,000건 총 시간 열을 추가해 출력해 줘

**5. 결과의 적용 범위 정리**  
일반화하면 안 되는 이유

이 결과가 통하는 경우와 통하지 않는 경우를 세 줄로 출력합니다. 적을 것은 없고, 실행해서 읽어 보면 됩니다.

정해진 보기 중에서 고르는 좁은 과제라서 sLM이 상용 API와 비슷하거나 앞설 수 있었습니다(강사 실행 94.4% 대 88.9%). 요약이나 글쓰기 같은 과제에서도 그렇다고 말할 수는 없습니다.

In [ ]:
print('통하는 자리 · 유형 여덟 개 중 고르기')
print('안 통하는 자리 · 요약, 추론, 글쓰기')
print('작은 모델은 좁은 과제에서 쓸 만합니다')
# Claude Code에 물어보기: 요약 과제에서 Gemma E2B와 상용 API를 비교할 때 쓸 평가 지표 알려 줘

**6. 1,000건 처리 비용 비교**  
토큰 비용과 GPU 시간 비용을 나란히 놓습니다

건당 토큰 수로 API의 1,000건 비용을, 건당 시간과 GPU 시간당 요금으로 sLM의 1,000건 비용을 계산합니다. 키 없이 실행했으면 강사가 실제로 쓴 토큰 수로 계산합니다. 정확도, 속도와 함께 비용까지 봐야 어느 쪽을 쓸지 정할 수 있습니다. sLM 비용은 건당 시간에 따라 공간마다 조금 다릅니다.

In [ ]:
# 실제 호출했으면 쓴 토큰, 아니면 강사 실측 토큰으로 계산합니다
if claude.live and claude.input_tokens:
    input_tokens = claude.input_tokens
    output_tokens = claude.output_tokens
else:
    input_tokens = facts['api']['tokens_in']
    output_tokens = facts['api']['tokens_out']
input_per_question = input_tokens / len(evalset)
output_per_question = output_tokens / len(evalset)

# 1,000건 비용. 단가: API는 100만 토큰당 달러, sLM은 RunPod RTX 4090 시간당 달러
INPUT_PRICE, OUTPUT_PRICE, GPU_PRICE_PER_HOUR = 4, 20, 0.89
api_cost = (input_per_question * INPUT_PRICE
            + output_per_question * OUTPUT_PRICE) / 1e6 * 1000
slm_cost = slm_result['seconds_per_question'] * 1000 / 3600 * GPU_PRICE_PER_HOUR

print('건당 토큰 · 입력', round(input_per_question), '· 출력', round(output_per_question))
print('API   $', round(api_cost, 2))
print('sLM   $', round(slm_cost, 2))
# Claude Code에 물어보기: api_cost와 slm_cost가 같아지는 건당 시간을 계산해 줘

**여기까지 확인**
- 같은 36문항으로 두 모델의 정확도를 비교했습니다
- 시간 차이를 봤다: 건당 초
- 비용을 비교했다: 1,000건 기준

## 예제 3-1. 좋아졌는지 확인합니다

**1. 전후 결과표 작성**  
학습 전후에 측정한 값을 그대로 씁니다

예제 1-1과 1-3에서 측정한 정확도와 형식 준수율을 전후 표로 출력합니다. 공간을 다시 켜서 그 값이 없으면 강사 실측값(`b6_facts.json`)을 씁니다. 숫자를 한 표에 놓으면 무엇이 얼마나 좋아졌는지 한눈에 보입니다.

In [ ]:
# 이 노트북에서 1-1과 1-3을 실행했으면 그 값을, 다시 연 뒤면 강사 실측값을 씁니다
if 'before_result' not in globals() or 'after_result' not in globals():
    facts = load_json('day06_data/b6_facts.json')
    before_result = facts_result(facts, 'before')
    after_result = facts_result(facts, 'after')

print('            전      후')
print(f"정확도    {before_result['accuracy']:6.1f} {after_result['accuracy']:6.1f}")
print(f"형식 준수 {before_result['format_rate']:6.1f} {after_result['format_rate']:6.1f}")
# Claude Code에 물어보기: facts_result()가 b6_facts.json의 어느 키를 읽는지 알려 줘

**2. 틀린 문항 확인**  
정확도를 건수로 환산하면 몇 건을 틀렸는지 나옵니다

학습 뒤에도 틀린 문항을 모아 몇 건인지와 앞 세 건을 출력합니다. 보고할 때는 정확도 몇 %보다 36건 가운데 몇 건이라고 말하는 편이 알아듣기 쉽습니다.

In [ ]:
# 학습 후 답이 정답과 다른 문항만 (답, 행)으로 모읍니다
wrong_rows = []
for answer, row in zip(after_result['answers'], evalset):
    if answer != row['a']:
        wrong_rows.append((answer, row))
print('틀린 건', len(wrong_rows), '/', len(evalset))

for answer, row in wrong_rows[:3]:
    print(' ', answer, '←', row['a'], '·', row['q'][:16])
# Claude Code에 물어보기: wrong_rows의 정답 유형별 건수를 label_counts 비율과 비교해 줘

**3. 좋아진 문항과 나빠진 문항 집계**  
점수가 올라도 학습 전에 맞던 문항을 틀리게 됐을 수 있습니다

학습 전후 답을 문항마다 비교해 좋아진 문항과 나빠진 문항 수를 집계하고, 나빠진 문항을 한 줄씩 출력합니다. 도입 여부를 판단하는 사람에게는 나빠진 문항도 함께 알려야 합니다.

In [ ]:
# compare_answers()는 학습 전후 답을 문항마다 비교합니다
improved, worsened = compare_answers(before_result['answers'],
                                     after_result['answers'], evalset)
print('좋아진 문항', len(improved), '· 나빠진 문항', len(worsened))

for row in worsened:
    print('  나빠짐 ·', row['a'], '·', row['q'][:16])
# Claude Code에 물어보기: compare_answers()가 improved와 worsened를 가르는 조건을 알려 줘

**4. 재현 조건 기록**  
점수 옆에 반드시 남기는 조건입니다

모델, 학습 설정, LoRA 설정, 시드와 GPU를 네 줄로 출력합니다. 점수만 적어 두면 다른 사람이 같은 결과를 다시 낼 수 없기 때문입니다.

In [ ]:
print('모델 ·', MODEL_NAME.split('/')[-1], '· bf16')
print('학습 ·', len(train_rows), '건 · 3에포크 · 학습률 2e-4')
print('LoRA · r=8 alpha=16 q,k,v,o')

gpu_name = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else DEVICE
print('시드 ·', SEED, '· GPU', gpu_name)
# Claude Code에 물어보기: 이 네 줄을 run_info.json으로 저장하는 코드 작성해 줘

**여기까지 확인**
- 정확도와 형식 준수율, 두 지표의 전후표를 만들었습니다
- 틀린 건을 열어 어떤 유형끼리 헷갈리는지 봤습니다
- 좋아진 문항과 나빠진 문항을 집계했고, 보고할 때는 둘 다 적습니다

## 예제 3-2. 학습 데이터는 몇 건이 필요할까요?

**1. 학습 데이터 25%로 재학습**  
25%와 50%로 같은 설정으로 학습합니다

학습 데이터 앞 25%(36건)만 써서 같은 설정으로 학습하고 정확도를 출력합니다. 데이터가 얼마나 있어야 학습 효과가 나는지 알면 데이터를 더 모을지 판단할 수 있습니다. 학습 전(예제 1-1)보다 오르는지 확인합니다.

In [ ]:
# 비교에 쓴 모델을 내리고 시작합니다
free_memory('tuned_model')

# 학습 건수 → 평가셋 정확도
accuracy_by_size = {}

# 학습 데이터 앞 25%로 같은 설정을 학습하고 측정합니다
subset_size = int(len(train_rows) * 0.25)
subset = train_rows[:subset_size]
small_model, _ = train_lora(subset, lora_rank=8, epochs=3, show_progress=False)
small_result = evaluate(small_model, evalset)
accuracy_by_size[len(subset)] = small_result['accuracy']
free_memory('small_model')

print('25%', len(subset), '건 ·', accuracy_by_size[len(subset)], '%')
# Claude Code에 물어보기: subset 앞 25%에 label_names 8종이 모두 있는지 확인해 줘

**2. 학습 데이터 50%로 재학습**  
72건, 설정은 같습니다

학습 데이터 앞 50%(72건)로 같은 학습을 한 번 더 합니다. 25%와 100% 사이에 점이 하나 더 있어야 어디서 오르기 시작하는지 보입니다.

In [ ]:
# 학습 데이터 앞 50%
subset_size = int(len(train_rows) * 0.5)
subset = train_rows[:subset_size]
small_model, _ = train_lora(subset, lora_rank=8, epochs=3, show_progress=False)
small_result = evaluate(small_model, evalset)
accuracy_by_size[len(subset)] = small_result['accuracy']
free_memory('small_model')

print('50%', len(subset), '건 ·', accuracy_by_size[len(subset)], '%')
# Claude Code에 물어보기: train_rows 앞 50% 대신 무작위 50%를 뽑을 때 시드 고정 방법 알려 줘

**3. 네 지점의 정확도 비교**  
학습 0건(프롬프트만), 25%, 50%, 100%를 한 줄로 봅니다

학습 전(0건)과 전체(144건) 정확도를 더해 네 지점을 건수 순서로 출력합니다. 구간마다 오른 폭을 보면 데이터를 더 모았을 때 더 오를지 판단할 수 있습니다. 강사 실행에서는 72건부터 크게 오르고 144건까지 계속 올랐습니다. 학습 결과는 실행마다 한두 문항씩 달라질 수 있으니 전체 흐름을 봅니다.

In [ ]:
# 0건은 예제 1-1, 100%는 예제 1-3에서 측정한 값입니다
accuracy_by_size[0] = before_result['accuracy']
accuracy_by_size[len(train_rows)] = after_result['accuracy']

for size in sorted(accuracy_by_size):
    print(f'{size:4d}건 {accuracy_by_size[size]:5.1f}%')
# Claude Code에 물어보기: accuracy_by_size 네 지점의 구간별 오름폭(%p)을 계산해 표시해 줘

**여기까지 확인**
- 네 지점의 정확도를 출력했습니다: 0, 25%, 50%, 100%
- 오름폭을 봤다: 어디서 크게 오르고 어디서 줄어드는지

## 부록 · 심화. train_lora() 안에서 일어나는 일

수업 본문에서는 `train_lora(train_rows, lora_rank=8, epochs=3)` 한 줄로 학습했습니다. 그 안의 학습 반복은 아래와 같습니다. `day06_data/day06_helpers.py`의 반복 부분을 옮기고 주석은 설명을 보태 고쳤습니다. 이 코드는 읽기만 하고 실행하지 않습니다. `order`, `optimizer` 같은 변수는 `train_lora()` 안에서만 만들어지므로 따로 실행하면 오류가 납니다.

```python
# 학습 데이터 전체를 epochs 번 되풀이합니다 (다른 주석도 '~합니다'로 통일)
    random.shuffle(order)                         # 에포크마다 순서를 섞습니다
    for begin in range(0, len(order), batch_size):
        batch = order[begin:begin + batch_size]   # 4건씩 묶음(배치)
        encoded = tokenizer([build_text(row) for row in batch], return_tensors='pt',
                            padding=True, truncation=True, max_length=128,
                            add_special_tokens=False).to(DEVICE)
        labels = encoded['input_ids'].clone()
        labels[encoded['attention_mask'] == 0] = -100   # 패딩 토큰은 손실에서 뺍니다
        loss = model(**encoded, labels=labels).loss       # 손실: 정답과 얼마나 다른가
        loss.backward()                                   # 기울기: 손실을 줄이는 방향
        optimizer.step()                                  # 옵티마이저: 그 방향으로 LoRA 값을 고침
        # 다음 배치를 위해 쌓인 기울기를 0으로 지웁니다
```

- **기울기**: 가중치마다 어느 쪽으로 얼마나 바꾸면 손실이 줄어드는지 계산한 값입니다.
- **옵티마이저(AdamW)**: 기울기와 학습률로 실제 값을 업데이트하는 규칙입니다. 학습률이 크면 한 번에 크게 업데이트합니다.
- LoRA 행렬만 `requires_grad`가 켜져 있어, 원본 가중치 51억 개는 업데이트하지 않습니다.

## 부록 · 심화. 지속 개선 루프

운영에서 틀린 건을 모아 정답을 붙이고 학습 데이터에 더해 다시 학습한 뒤, 같은 평가셋으로 다시 측정하는 주기입니다.

1. 운영에서 틀린 건을 모읍니다 / 2. 사람이 정답을 붙입니다 / 3. 학습 데이터에 더합니다 / 4. 다시 학습하고 같은 평가셋으로 재측정합니다
2. 사람이 정답을 붙인다
3. 학습 데이터에 더한다
4. 다시 학습하고 같은 평가셋으로 재측정

## 부록 · 심화. QLoRA로 학습해 보기

원본을 4비트(NF4)로 줄여 올리고 그 위에서 LoRA만 학습합니다. `train_lora()`에 `quantization_config` 하나만 더 넘깁니다. 예제 1-3의 bf16 LoRA와 같은 데이터, 같은 설정이라, 정확도와 GPU에 잡힌 메모리를 나란히 비교할 수 있습니다. GPU(cuda)에서만 됩니다.

학습 시간, 학습 중 GPU에 잡힌 가장 큰 메모리, 평가셋 정확도를 출력합니다. 강사 실행에서는 최대 메모리가 9.2GB(bf16 LoRA는 12.3GB)였고, 정확도는 94.4%로 bf16 LoRA와 같았습니다. 모델의 임베딩 표(약 5GB)는 4비트로 줄지 않아서, 메모리가 계산값(2.4GB)만큼 줄지는 않습니다.

In [ ]:
# 부록 QLoRA: 원본을 nf4 4비트로 올리고 LoRA만 학습합니다 (GPU 필요, 1분쯤)
free_memory('tuned_model', 'model', 'base_model')
torch.cuda.reset_peak_memory_stats()        # 최대 메모리 기록을 새로 시작합니다

qlora_model, qlora_history = train_lora(train_rows, lora_rank=8, epochs=3,
                                        show_progress=False,
                                        quantization_config=quantization_for('nf4'))
trained_million = round(qlora_history['trainable_params'] / 1e6, 2)
print('학습 시간', qlora_history['seconds'], '초 · 학습한 파라미터', trained_million, 'M')
peak_gb = torch.cuda.max_memory_allocated() / 1024**3
print('학습 중 가장 큰 GPU 메모리', round(peak_gb, 1), 'GB')

qlora_result = evaluate(qlora_model, evalset)
print('평가셋 정확도', qlora_result['accuracy'], '% (예제 1-3 bf16 LoRA 94.4%)')
free_memory('qlora_model')
# Claude Code에 물어보기: quantization_for('nf4')가 만드는 BitsAndBytesConfig 설정을 알려 줘

## 부록 · 심화. 현업 도구로 같은 학습을 한다면(PEFT, TRL)

오늘 `train_lora()`는 학습 반복을 직접 써서 내부 동작을 보여 줬습니다. 현업에서는 Hugging Face의 PEFT 라이브러리로 LoRA를 붙이고, TRL 라이브러리의 `SFTTrainer`로 같은 반복을 실행하는 경우가 많습니다. 선호 학습(DPO)은 같은 라이브러리의 `DPOTrainer`로, 선호하는 답과 선호하지 않는 답의 쌍을 데이터로 넣어 실행합니다. 아래는 읽기용 코드이고 실행하지 않습니다(오늘 노트북 환경에는 TRL을 설치하지 않았습니다).

```python
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

lora_config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, task_type='CAUSAL_LM',
                         target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'])
trainer = SFTTrainer(model=MODEL_NAME, train_dataset=train_dataset, peft_config=lora_config,
                     args=SFTConfig(num_train_epochs=3, learning_rate=2e-4, per_device_train_batch_size=4))
trainer.train()
```

- `train_dataset`은 질문과 정답 쌍을 한 줄에 한 건씩 담은 데이터셋입니다. 오늘의 `train_rows`와 같은 내용입니다.
- 교사 모델(정답을 붙여 주는 큰 모델)의 답을 정답 자리에 넣어 같은 방식으로 학습하면 증류가 됩니다. 실습 미션 1은 Claude 정규화 결과를 정답으로 넣어 학습하므로 이 방식에 가깝습니다. 정답 품질은 같은 평가셋으로 다시 측정해서 확인합니다.

## 부록. 공간이 꺼졌을 때 복구하기

커널을 다시 시작했거나, 멈춘 실습 공간을 다시 켰을 때 쓰는 셀입니다. 작업 폴더(/workspace)에 둔 파일은 그대로 있고 메모리에 올린 변수와 모델만 사라집니다.

아래 셀보다 먼저 환경 준비 셀과 준비 셀을 실행합니다. 두 셀은 이름이 비슷하지만 서로 다른 셀이라, 하나만 실행하면 아래 셀에서 오류가 납니다. 두 셀을 모두 실행한 다음 아래 셀을 실행하면 예제 1-3을 마친 상태로 돌아옵니다.

데이터와 프롬프트를 다시 만들고 저장한 어댑터를 붙여 모델을 올립니다. 다시 학습하지는 않습니다. 「어댑터 복원 완료」 한 줄이 나오면 예제 1-4부터 이어서 실행합니다.

In [ ]:
# 데이터와 프롬프트를 다시 만들고, 저장한 어댑터를 붙입니다 (다시 학습하지 않습니다)
from peft import PeftModel

chunks = load_json('day06_data/chunks.json')
documents = {}
for chunk in chunks:
    documents.setdefault(chunk['doc_id'], chunk)
label_names = sorted({chunk['type'] for chunk in chunks})
all_rows, train_rows, evalset = make_title_rows(documents, label_names)

prompt_template = ('다음 사내 문서 제목이 어느 유형인지 한 낱말로만 답하라.\n'
                   '유형: ' + ' / '.join(label_names) + '\n\n제목: {q}\n유형:')
set_task(prompt_template, label_names)

# 저장한 어댑터가 없으면(예제 1-3 (5) 전이면) 예제 1-1부터 차례로 실행합니다
assert os.path.isdir('lora_doctype'), (
    'lora_doctype 폴더가 없습니다 · 예제 1-1 부터 차례로 실행하세요')
free_memory('tuned_model')
tuned_model = PeftModel.from_pretrained(load_model(), 'lora_doctype')
print('어댑터 복원 완료 · 다시 학습하지 않았습니다')
# Claude Code에 물어보기: lora_doctype이 없을 때 assert 대신 안내만 출력하도록 수정해 줘